<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/%EC%9B%B9%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EA%B8%B0%EB%A7%90%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import gradio as gr

# 1. 데이터 로드
file_path = 'AdventureWorks Sales.xlsx'
df_sales = pd.read_excel(file_path, sheet_name='Sales_data')
df_prod = pd.read_excel(file_path, sheet_name='Product_data')
df_cust = pd.read_excel(file_path, sheet_name='Customer_data')
#판매 데이터,상품 데이터,고객 데이터
#Sales = 행동,Product = 대상,Customer = 주체

# 2. 전처리 (통합 데이터셋)
df = pd.merge(df_sales, df_cust, on='CustomerKey', how='left')
df = pd.merge(df, df_prod, on='ProductKey', how='left')
df = df.dropna(subset=['Product', 'Country-Region', 'Sales Amount', 'City'])

# 3. 모델별 독립적인 인코딩 및 학습
# [분류 모델용]
le_city = LabelEncoder()
df['City_enc'] = le_city.fit_transform(df['City'].astype(str))
df['Buy'] = (df['Sales Amount'] > df['Sales Amount'].median()).astype(int)
clf = RandomForestClassifier(random_state=42).fit(df[['City_enc']], df['Buy'])

# [회귀 모델용]
le_prod = LabelEncoder()
le_reg = LabelEncoder()
df['P_enc'] = le_prod.fit_transform(df['Product'].astype(str))
df['R_enc'] = le_reg.fit_transform(df['Country-Region'].astype(str))
reg = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42).fit(df[['P_enc', 'R_enc']], df['Sales Amount'])

# 4. 개별 예측 함수
def predict_crm(city):
    if city not in le_city.classes_: return "정보 없음", ""
    enc = le_city.transform([city])[0]
    prob = clf.predict_proba([[enc]])[0][1]
    return f"{'Buy' if prob > 0.5 else 'Not Buy'}", f"{prob*100:.1f}%"

def predict_sales(product, region):
    if product not in le_prod.classes_ or region not in le_reg.classes_: return "$0.00"
    p = le_prod.transform([product])[0]
    r = le_reg.transform([region])[0]
    pred = reg.predict([[p, r]])[0]
    # 지역별 가중치 적용 (회귀 성능 강화)
    region_bias = df[df['Country-Region'] == region]['Sales Amount'].mean() / df['Sales Amount'].mean()
    final_pred = pred * (0.8 + 0.2 * region_bias)
    return f"${final_pred:,.2f}"

# 5. Gradio 통합 UI
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🚀 AdventureWorks 통합 분석 시스템")
    with gr.Tab("고객 구매 예측"):
        city_input = gr.Dropdown(sorted(df['City'].unique().tolist()), label="고객 도시")
        out1 = gr.Textbox(label="예측 결과"); out2 = gr.Textbox(label="구매 확률")
        gr.Button("분석 실행").click(predict_crm, [city_input], [out1, out2])
    with gr.Tab("판매량 회귀 예측"):
        prod_input = gr.Dropdown(sorted(df['Product'].unique().tolist()), label="상품명")
        reg_input = gr.Dropdown(sorted(df['Country-Region'].unique().tolist()), label="지역")
        out3 = gr.Textbox(label="예상 판매 금액")
        gr.Button("예측 실행").click(predict_sales, [prod_input, reg_input], out3)

demo.launch(share=True)

/tmp/ipykernel_13470/2319332035.py:50: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59c59fa5f98ed921c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
